This page shows the latest publications (in descending order of publication date) from all of the open access publishers in the [ScholarLed](https://scholarled.org/) consortium ([Mattering Press](https://www.matteringpress.org/), [meson press](https://meson.press/), [Open Book Publishers](https://www.openbookpublishers.com/), [punctum books](https://punctumbooks.com/), [African Minds](https://www.africanminds.co.za/), and [mediastudies.press](https://www.mediastudies.press/)).

Metadata is licensed as [Creative Commons Zero (CC0)](https://creativecommons.org/share-your-work/public-domain/cc0/) and is retrieved from [Thoth](https://thoth.pub/)'s open APIs.

In [1]:
from datetime import datetime

# datetime object containing current date and time
now = datetime.now()

# YY-mm-dd H:M:S
now_string = now.strftime("%Y-%m-%d %H:%M:%S")

print('Last updated: ' + now_string)

Last updated: 2026-06-01 11:51:40


In [ ]:
from thothlibrary import ThothClient
from datetime import datetime
import json

# publisher ID variables
mattering = '17d701c1-307e-4228-83ca-d8e90d7b87a6'
meson = 'f0ae98da-c433-45b8-af3f-5c709ad0221b'
obp = '85fd969a-a16c-480b-b641-cb9adf979c3b'
punctum = '9c41b13c-cecc-4f6a-a151-be4682915ef5'
african = 'b61217e4-3134-4bfe-8695-30e047ed3f57'
media = '4ab3bec2-c491-46d4-8731-47a5d9b33cc5'

publishers_ids = '["' + mattering + '" "' + meson + '" "' + obp + '" "' + punctum + '" "' + african + '" "' + media + '"]'
 
# calling the Thoth GraphQL API
thoth = ThothClient()
thoth.QUERIES['books']['fields'].append('coverUrl')

thoth.QUERIES['books']['fields'] = [
    f.replace('markupFormat: JATS_XML', 'markupFormat: MARKDOWN') 
    if 'abstracts' in f else f 
    for f in thoth.QUERIES['books']['fields']
]

response = thoth.books(
    publishers=publishers_ids,
    work_status='ACTIVE',
    order='{field: PUBLICATION_DATE, direction: DESC}'
)

months = []

for result in response:
    # get fullTitle from nested titles list
    titles = result.get('titles', [])
    full_title = titles[0]['fullTitle'] if titles else 'Untitled'

    if result.get('publicationDate'):
        date = datetime.strptime(result['publicationDate'], '%Y-%m-%d')
        date = date.strftime('%B %Y')
        if not months or date != months[-1]:
            print('## ' + date + '\n')
            months.append(date)
    else:
        print('## No date\n')

    print('### ' + full_title + '\n')

    if result.get('coverUrl'):
        print('<img src="' + result['coverUrl'] + '" alt="cover for ' + full_title + '" width="300"/>\n')

    for contribution in result.get('contributions', []):
        print(contribution['contributionType'].capitalize().replace("_", " ") + ': ' + contribution['fullName'] + '\n')

    if result.get('place') and result['imprint']['publisher']['publisherName'] and result.get('publicationDate'):
        print(result['place'] + ': ' + result['imprint']['publisher']['publisherName'] + ', ' + result['publicationDate'][0:4] + '\n')

    if result.get('doi'):
        print('[' + result['doi'] + '](' + result['doi'] + ')\n')

    # shortAbstract replaced by abstracts list; using longAbstract
    abstracts = result.get('abstracts', [])
    if abstracts:
        print(abstracts[0]['content'] + '\n')

    print('\n\n')

ModuleNotFoundError: No module named 'thothlibrary'